# Inverse Problems 2026 - Exercise 3 (home exercises)

Return by **Fri 20.11.2026 at 23:59** in Moodle! You can ask for help with these exercises in the exercise session the same day (10:00, room 7334).

The returned exercises should be renamed `Exercise03_lastname_firstname.ipynb` and contain your solutions in clear and readable manner and format. The programming exercises need to be runnable with minimal effort. If you find markdown too cumbersome for writing the necessary math, you can attach **a single extra pdf-file** for the theoretical exercises and denote it `Exercise03_lastname_firstname.pdf`.

___
In the following exercises you should use the given function `create_sinogram(N, angles)` to create the simulated tomography data, formulated as sinogram.
You can create the forward operator as matrix `A` using the function `create_radon_matrix(N, n_det, angles, sparse=False)`.
___

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.transform import resize
import astra

def create_sinogram(N, angles = np.arange(360)):
    """Simple data generation function. Given the resolution `N` (for N x N images),
    creates the famous Shepp-Logan phantom at TWICE the spatial resolution, then projects
    it to noise-free sinogram using the given projection `angles` (in degrees) to avoid
    inverse crime.
    Shepp-Logan phantom of size N x N is also returned
    
    If `angles` are not specified, use 360 projections at 0 degree interval [0, 1, ..., 359].
    """
    n_det = 2*np.ceil(np.sqrt(2)*N).astype(int)
    anglesRad = np.deg2rad(angles)

    projGeom = astra.create_proj_geom('parallel', 0.5, n_det, anglesRad)
    volGeom = astra.create_vol_geom(N,N)
    projId = astra.create_projector("strip", projGeom, volGeom)

    volId, phantom_big = astra.data2d.shepp_logan(volGeom, modified=True)
    phantom = resize(phantom_big, output_shape=(N, N))

    # Calculate forward projection
    projection_id = astra.data2d.create('-sino', projGeom)
    cfg = astra.astra_dict('FP')
    cfg['ProjectorId'] = projId
    cfg['ProjectionDataId'] = projection_id
    cfg['VolumeDataId'] = volId
    algorithm_id = astra.algorithm.create(cfg)

    astra.algorithm.run(algorithm_id)

    sino_big = astra.data2d.get(projection_id)

    # Bin along detector positions -> detector width halves
    sino = 0.5*(sino_big[:, 0:-2:2] + sino_big[:, 1:-1:2]) / N
    return sino, phantom

def create_radon_matrix(N, n_det, angles):
    """Helper function to create the forward operator as a matrix using the
    ASTRA-toolbox.
    
    N is the target resolution, i.e. it is assumed to be N x N image dropped to vector along
    rows (C-format, default behaviour in Numpy).
    n_det is the number of detector elements or rows in the sinogram.
    """
    anglesRad = np.deg2rad(angles)
    projGeom = astra.create_proj_geom('parallel', 1.0, n_det, anglesRad)
    volGeom = astra.create_vol_geom(N,N)
    projId = astra.create_projector("strip", projGeom, volGeom)
    matrixId = astra.projector.matrix(projId)
    astra.projector.delete(projId) # Free memory
    A = astra.matrix.get(matrixId)
    return A / N

N = 64
n_ang = 360
m, x_true = create_sinogram(N, np.arange(n_ang))
n_ang, n_det = m.shape

A = create_radon_matrix(N, n_det, np.arange(n_ang))
m_crime = (A @ x_true.reshape((-1,1))). reshape((n_ang, n_det))
bp = (A.T @ m.reshape((-1,1))).reshape((N,N))

fig = plt.figure(figsize=(8,5))

ax2 = fig.add_subplot(2,1,2)
im2 = ax2.imshow(m.T, aspect=2)
fig.colorbar(im2)
ax2.set_title("Sinogram");

ax1 = fig.add_subplot(2,2,1)
im1 = ax1.imshow(x_true)
fig.colorbar(im1)
ax1.set_title("Shepp-Logan phantom");

ax3 = fig.add_subplot(2,2,2)
im3 = ax3.imshow(bp)
fig.colorbar(im3)
ax3.set_title("Naive backprojection");

## Task 1 (Filtered backprojection)
Theorem 3.3 of the lecture notes gives the analytic Radon inversion formula which is also the basis for the industry standard Filtered Backprojection algorithm (FBP):
$$ f = \frac{1}{4\pi} \mathcal{R}^* \mathcal{I}^{-1} \mathcal{R} f. $$
$\mathcal{R} f$ is the measured data, the adjoint of Radon transform $\mathcal{R}^*$ is the backprojection and the filtering means the inverse Ries potential which is a specific Fourier domain multiplier.

1.) Let's implement FBP ourselves. Below you can find partial code. Finish the implementation of the inverse Riesz potential and apply `FBP` to given sinogram data. Note that the sinogram should not be dropped to a vector!  
2.) Test our implementation of FBP on data with noise levels $\sigma = 0, 0.01$ and $0.05$. What do you see?  
3.) Here the Fourier multiplier or filter is defined as $h_1(s) = |s|$, which is in theory the correct choice. It is also known as the "ramp" filter. In practice [other similar filters](https://scikit-image.org/docs/stable/auto_examples/transform/plot_radon_transform.html#reconstruction-with-the-filtered-back-projection-fbp) are used. Implement and test the Shepp-Logan filter $h_2(s) = |s| \tfrac{\sin(0.5 \pi s)}{0.5 \pi s} = h_1(s) \tfrac{\sin(0.5 \pi s)}{0.5 \pi s}$, proposed by [Larry Shepp and Benjamin Logan in 1974](https://web.archive.org/web/20160304035526/http://stat.wharton.upenn.edu/~shepp/publications/33.pdf) again using the noise levels $\sigma = 0, 0.01$ and $0.05$. What do you see and why? 

_Note: Getting the scale of the reconstructed values correct is difficult but you do not have to worry about that here._

In [ ]:
"""
Responsibility of this cell:
Reconstruct an N*N image by filtered backprojection.

FFT each sinogram row along the detector axis, apply the chosen frequency
filter, and inverse FFT. Scale using the assignment's discrete convention,
then flatten the filtered sinogram and backproject with BP_mtx (A.T).
"""
from scipy.fft import fft, ifft, fftfreq, fftshift
rng = np.random.default_rng(42)

def ramp(s):
    """Inverse Riesz multiplier h(s) = |s| at the FFT frequency bins"""
    return np.abs(s)

def shepp_logan(s):
    """Ramp multiplied by sinc(s/2) = sin(pi*s/2)/(pi*s/2)"""
    # np.sinc(0) = 1, so the zero-frequency value is handled without division
    return np.abs(s) * np.sinc(s / 2)

def FBP(sinogram, N, BP_mtx, filter=ramp):
    """Filter each projection along its detector axis, then backproject"""
    n_ang, n_det = sinogram.shape
    if BP_mtx.shape != (N * N, n_ang * n_det):
        raise ValueError("Backprojection matrix and sinogram dimensions do not match")

    # Step 1: Transform along detector positions, not projection angles
    freq_data = fft(sinogram, axis=1)

    # Step 2: Apply the inverse Riesz multiplier at the FFT bins
    # fftfreq lies in [-0.5, 0.5) cycles/sample
    freqs = fftfreq(n_det).reshape((1, -1))
    freq_data *= filter(freqs)
    freq_data *= n_det**2 / n_ang  # Provided discrete scaling convention.

    # Step 3: Return to detector space (numerical imaginary roundoff is discarded)
    sino_data = ifft(freq_data, axis=1).real

    # Step 4: A.T is the discrete backprojection (only now flatten the sinogram)
    recn = BP_mtx @ sino_data.reshape(-1)
    return np.asarray(recn).reshape((N, N))


In [ ]:
"""
Responsibility of this cell:
Compare Ramp and Shepp–Logan FBP at three measurement-noise levels.

Use the same seeded noisy sinogram for both filters at each level. Plot
the reconstructions on a shared grayscale, report relative errors after
correcting only for global intensity scale, and plot the filter responses.
"""
# Step 1: Use one reproducible noise realization per level for both filters
noise_levels = (0, 0.01, 0.05)
noisy_sinograms = {
    sigma: m + sigma * rng.standard_normal(m.shape) for sigma in noise_levels
}

# Step 2: Reconstruct with the discrete adjoint of the supplied Radon matrix
filters = {"Ramp": ramp, "Shepp-Logan": shepp_logan}
reconstructions = {
    (name, sigma): FBP(noisy_sinograms[sigma], N, A.T, multiplier)
    for name, multiplier in filters.items()
    for sigma in noise_levels
}

# Step 3: Use identical display limits so noise and filter differences are comparable
clean_ramp = reconstructions["Ramp", 0]
vmin, vmax = np.percentile(clean_ramp, (1, 99))
fig, axes = plt.subplots(2, 3, figsize=(12, 8), constrained_layout=True)
for row, name in enumerate(filters):
    for col, sigma in enumerate(noise_levels):
        image = reconstructions[name, sigma]
        im = axes[row, col].imshow(image, cmap="gray", vmin=vmin, vmax=vmax)
        axes[row, col].set_title(f"{name}, σ = {sigma:g}")
        axes[row, col].axis("off")
fig.colorbar(im, ax=axes, shrink=0.78, label="Reconstructed intensity")
plt.show()

# Step 4: Compare spatial error after fitting one display-scale factor
# The assignment explicitly allows a global scale mismatch
# Fitting it is for evaluation only and does not alter any reconstruction or plotted image
print("Scale-adjusted relative error against supplied phantom:")
for name in filters:
    for sigma in noise_levels:
        image = reconstructions[name, sigma]
        gain = np.vdot(image, x_true) / np.vdot(image, image)
        relative_error = np.linalg.norm(gain * image - x_true) / np.linalg.norm(x_true)
        print(f"  {name:12s} σ={sigma:4.2f}: {relative_error:.4f}")

# The two Fourier multipliers clarify the expected noise behavior
frequency = np.linspace(0, 0.5, 300)
plt.figure(figsize=(7, 3.5))
plt.plot(frequency, ramp(frequency), label="Ramp")
plt.plot(frequency, shepp_logan(frequency), label="Shepp-Logan")
plt.xlabel("Detector frequency (cycles/sample)")
plt.ylabel("Filter multiplier")
plt.title("FBP filter response on the FFT frequency range")
plt.legend()
plt.grid(alpha=0.25)
plt.show()



### Task 1: Results and Interpretation

Filtered backprojection produces sharper images than unfiltered backprojection. As the noise level increases, the reconstructions become noisier because the filters amplify high-frequency components.

The Ramp filter preserves slightly more detail, while Shepp–Logan produces smoother results by reducing high frequencies. Ramp performs slightly better at σ = 0 and 0.01, but Shepp–Logan gives a lower reconstruction error at σ = 0.05 (0.6259 vs. 0.6359).


___
## Task 2 (Tikhonov regularization)
Using the given forward matrix `A` and noise-free data vector `m`, add normally distributed noise with standard deviation $\sigma = 0.02$ to it. Then solve the Tikhonov regularized solution for regularization parameters $\alpha = 0.01, 0.1$ and $1$ and plot the result.

In [ ]:
"""
This cell's responsibility:
Compare Tikhonov reconstructions for three regularization strengths.

Add reproducible Gaussian noise to the sinogram, then use LSMR to solve
the regularized least-squares problem for each alpha. Report fit and
solver checks, and plot the reconstructions beside the true phantom
on a shared grayscale.
"""
from scipy.sparse.linalg import lsmr

# Step 1: Form a reproducible noisy data vector in A's row ordering
sigma_tikhonov = 0.02
rng_tikhonov = np.random.default_rng(43)
m_vector = m.ravel(order="C")
y = m_vector + sigma_tikhonov * rng_tikhonov.standard_normal(m_vector.shape)
assert A.shape == (y.size, N * N)

# Step 2: Solve ||A x - y||^2 + alpha ||x||^2 for each required alpha
alphas = (0.01, 0.1, 1.0)
tikhonov_reconstructions = {}
tikhonov_results = {}
for alpha in alphas:
    result = lsmr(A, y, damp=np.sqrt(alpha), atol=1e-10, btol=1e-10, maxiter=2000)
    x_alpha = result[0]
    tikhonov_reconstructions[alpha] = x_alpha.reshape((N, N))
    residual = np.linalg.norm(A @ x_alpha - y)
    solution_norm = np.linalg.norm(x_alpha)
    normal_error = np.linalg.norm(A.T @ (A @ x_alpha - y) + alpha * x_alpha)
    tikhonov_results[alpha] = (residual, solution_norm, normal_error, result[2], result[1])
    assert np.isfinite(x_alpha).all()

# Step 3: Report the fit, regularization effect, and normal-equation check
print(f"Noise standard deviation: {sigma_tikhonov:.2f}; data vector length: {y.size}")
print("alpha   ||Ax-y||_2   ||x||_2   normal-equation error   iterations   stop code")
for alpha in alphas:
    residual, solution_norm, normal_error, iterations, stop_code = tikhonov_results[alpha]
    print(f"{alpha:5.2f}   {residual:10.4f}   {solution_norm:8.4f}   "
          f"{normal_error:21.2e}   {iterations:10d}   {stop_code:9d}")

# Step 4: Compare reconstructions on the same grayscale scale as the phantom
fig, axes = plt.subplots(1, 4, figsize=(14, 4), constrained_layout=True)
for ax, image, title in zip(
    axes,
    (x_true, *(tikhonov_reconstructions[alpha] for alpha in alphas)),
    ("True phantom", *(f"Tikhonov α = {alpha:g}" for alpha in alphas)),
):
    im = ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.8, label="Image intensity")
plt.show()


___
## Task 3 (Iterative regularization)
Write a routine that implements the Landweber iterative method discussed in the lectures for $k = 2,7,25$ and $50$ iterations. Use $\beta = 1 / {\lambda^2_1}$ and some level of measurement noise. _Hint:_ For estimating $\lambda_1$, consider the [spectral norm](https://en.wikipedia.org/wiki/Matrix_norm#Spectral_norm_(p_=_2)) of $A$.

In [ ]:
"""
This cell's responsibility:
Compare Landweber reconstructions at four iteration counts.

Estimate the largest singular value of A to set the prescribed step
size, then apply Landweber iteration to Task 2's noisy data. Save the
iterates at k = 2, 7, 25, and 50, report their data residuals, and
plot them on a shared grayscale.
"""
from scipy.sparse.linalg import svds

# Step 1: Estimate the spectral norm of the sparse forward matrix
lambda_1 = svds(
    A, k=1, which="LM", return_singular_vectors=False,
    rng=np.random.default_rng(44),
)[0]
beta = 1.0 / lambda_1**2

# Step 2: Apply Landweber iteration to Task 2's noisy data (sigma = 0.02)
def landweber(A, data, step_size, checkpoints):
    """Return selected iterates of x <- x + step_size A.T (data - A x)."""
    x = np.zeros(A.shape[1])
    saved = {}
    for k in range(1, max(checkpoints) + 1):
        x += step_size * (A.T @ (data - A @ x))
        if k in checkpoints:
            saved[k] = x.copy()
    return saved

iterations = (2, 7, 25, 50)
landweber_reconstructions = landweber(A, y, beta, iterations)

# Step 3: Check the requested iteration counts and report data residuals.
print(f"Measurement noise: sigma = {sigma_tikhonov:.2f}")
print(f"Largest singular value: lambda_1 = {lambda_1:.6f}")
print(f"Landweber step size: beta = {beta:.6f}")
for k in iterations:
    x_k = landweber_reconstructions[k]
    residual = np.linalg.norm(A @ x_k - y)
    print(f"k = {k:2d}: ||A x_k - y||_2 = {residual:.4f}")

# Step 4: Show all four reconstructions on a common grayscale scale
fig, axes = plt.subplots(1, 4, figsize=(14, 4), constrained_layout=True)
for ax, k in zip(axes, iterations):
    image = landweber_reconstructions[k].reshape((N, N))
    im = ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"Landweber, k = {k}")
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.8, label="Image intensity")
plt.show()


### Task 3

Using Task 2's noisy data ($\sigma=0.02$) and $x^0=0$, the sparse spectral-norm estimate is $\lambda_1=2.330253$, giving $\beta=1/\lambda_1^2=0.184159$. The update is $x^{k+1}=x^k+\beta A^T(y-Ax^k)$.

At $k=2$ the image is strongly blurred; the main structures emerge by $k=7$. Edges become clearer at $k=25$ and $50$, while fine-grained noise also becomes more visible. The data residual decreases from $6.7781$ at $k=2$ to $4.1081$ at $k=50$. Thus more iterations improve the fit, but early stopping limits noise amplification.


___
## Task 4 (TV denoising)
Total variation denoising (the Rudin–Osher–Fatemi model) finds
$$ f_\alpha = \operatorname{arg\,min}_f \left\lbrace \frac{1}{2} \| f - g^\delta \|^2 + \alpha \sum_{i,j} |(\nabla f)_{i,j}| \right\rbrace, $$
where $|(\nabla f)_{i,j}|$ is the Euclidean norm of the discrete gradient at pixel $(i,j)$. The objective is not differentiable where $\nabla f = 0$, so we replace $|\cdot|$ by the Huber function $\phi_\varepsilon$ (Exercise 2, Task 4). Gradient descent for the smoothed problem reads
$$ f^{k+1} = f^k + \tau \left( (g^\delta - f^k) - \alpha \nabla^T (w^k \nabla f^k) \right), \quad w^k_{i,j} = \frac{1}{\max \lbrace \varepsilon, |(\nabla f^k)_{i,j}| \rbrace}. $$

1. Implement the discrete gradient `grad` and its adjoint `grad_T` below. You should check that $\langle \nabla f, p \rangle = \langle f, \nabla^T p \rangle$ for random $f$ and $p$.
2. Implement the gradient descent and denoise the noisy image. Use $\varepsilon = 0.01$, step size $\tau = 1 / (1 + 8 \alpha / \varepsilon)$ and try a few values of $\alpha$.
3. Compare to the quadratic regularization $\frac{1}{2} \| f - g^\delta \|^2 + \frac{\alpha}{2} \| \nabla f \|^2$ (Tikhonov with $L = \nabla$). What happens to the edges?

In [ ]:
"""
This cell's responsibility:
Prepare the noisy image and discrete gradient operators for denoising.

Resize the Shepp–Logan phantom, add reproducible Gaussian noise, and
display it. Define forward differences with zero differences at the
last row and column, together with their exact adjoint.
"""
from skimage.data import shepp_logan_phantom
from skimage.transform import resize

rng = np.random.default_rng(3)
f_true = resize(shepp_logan_phantom(), (128, 128)) # Or use your own image from Exercise 1
g = f_true + 0.1 * rng.standard_normal(f_true.shape)

def grad(f):
    """Forward differences (h = 1) with zero difference at the last row/column (Neumann boundary).
    Returns an array of shape (2, N, M)."""
    # Step 1: Forward differences down rows and across columns.
    out = np.zeros((2, *f.shape), dtype=float)
    out[0, :-1, :] = f[1:, :] - f[:-1, :]
    out[1, :, :-1] = f[:, 1:] - f[:, :-1]
    return out

def grad_T(p):
    """Adjoint of `grad`, i.e. the negative (backward) divergence of the vector field p of shape (2, N, M)."""
    # Step 2: Add the transpose contributions of each valid difference.
    out = np.zeros(p.shape[1:], dtype=float)
    out[:-1, :] -= p[0, :-1, :]
    out[1:, :] += p[0, :-1, :]
    out[:, :-1] -= p[1, :, :-1]
    out[:, 1:] += p[1, :, :-1]
    return out

plt.imshow(g, cmap="gray")
plt.title("Noisy image");

In [ ]:
# Step 3: Verify the discrete adjoint identity, including boundary entries.
rng_adjoint = np.random.default_rng(4)
f_test = rng_adjoint.standard_normal((17, 19))
p_test = rng_adjoint.standard_normal((2, 17, 19))
Df = grad(f_test)
DTp = grad_T(p_test)
lhs = np.sum(Df * p_test)
rhs = np.sum(f_test * DTp)
relative_adjoint_error = abs(lhs - rhs) / max(1.0, abs(lhs), abs(rhs))
assert Df.shape == (2, 17, 19) and DTp.shape == (17, 19)
assert np.all(Df[0, -1, :] == 0) and np.all(Df[1, :, -1] == 0)
assert relative_adjoint_error < 1e-12
print(f"Adjoint relative error: {relative_adjoint_error:.2e}")


In [ ]:
"""
This cell's responsibility:
Denoise the image with Huber-smoothed total variation.

For three alpha values, use the prescribed step size and iterate until
the gradient is small. Check that each result is finite and lowers the
objective, then report its step size and iteration count.
"""
# Step 4: Minimize the Huber-smoothed TV objective by the assigned update.
def huber_tv_objective(f, noisy, alpha, epsilon):
    magnitude = np.sqrt(np.sum(grad(f)**2, axis=0))
    huber = np.where(magnitude <= epsilon,
                     magnitude**2 / (2 * epsilon), magnitude - epsilon / 2)
    return 0.5 * np.sum((f - noisy)**2) + alpha * np.sum(huber)

def tv_denoise(noisy, alpha, epsilon=0.01, tolerance=1e-5, max_updates=5000):
    f = noisy.copy()
    tau = 1.0 / (1.0 + 8.0 * alpha / epsilon)
    updates = 0
    for _ in range(max_updates):
        Df = grad(f)
        magnitude = np.sqrt(np.sum(Df**2, axis=0))
        weight = 1.0 / np.maximum(epsilon, magnitude)
        direction = (noisy - f) - alpha * grad_T(weight[None, :, :] * Df)
        gradient_rms = np.sqrt(np.mean(direction**2))
        if gradient_rms < tolerance:
            break
        f += tau * direction
        updates += 1
    return f, tau, updates, gradient_rms

epsilon = 0.01
alpha_values = (0.05, 0.1, 0.2)
tv_images = {}
print("Huber TV: alpha, tau, updates, final gradient RMS")
for alpha in alpha_values:
    image, tau, updates, gradient_rms = tv_denoise(g, alpha, epsilon)
    assert np.isfinite(image).all() and gradient_rms < 1e-5
    assert huber_tv_objective(image, g, alpha, epsilon) < huber_tv_objective(g, g, alpha, epsilon)
    tv_images[alpha] = image
    print(f"  {alpha:4.2f}  {tau:.6f}  {updates:4d}  {gradient_rms:.2e}")


In [ ]:
"""
This cell's responsibility:
Compare quadratic denoising with Huber TV at the chosen alpha values.

Solve each quadratic normal equation with conjugate gradients and check
its residual. Plot the noisy image, true image, and both sets of results
on a shared intensity scale.
"""

from scipy.sparse.linalg import LinearOperator, cg

# Step 5: Solve (I + alpha grad_T grad) f = g for quadratic regularization.
quadratic_images = {}
for alpha in alpha_values:
    def normal_matvec(vector):
        image = vector.reshape(g.shape)
        return (image + alpha * grad_T(grad(image))).ravel()

    operator = LinearOperator((g.size, g.size), matvec=normal_matvec, dtype=float)
    solution, info = cg(operator, g.ravel(), rtol=1e-10, atol=0, maxiter=1000)
    relative_residual = np.linalg.norm(operator @ solution - g.ravel()) / np.linalg.norm(g)
    assert info == 0 and relative_residual < 1e-8
    quadratic_images[alpha] = solution.reshape(g.shape)
    print(f"Quadratic alpha={alpha:.2f}: relative normal-equation residual={relative_residual:.2e}")

# Step 6: Compare both penalties on the same intensity scale.
fig, axes = plt.subplots(2, 4, figsize=(14, 7), constrained_layout=True)
images = ([g] + [tv_images[alpha] for alpha in alpha_values],
          [f_true] + [quadratic_images[alpha] for alpha in alpha_values])
titles = (["Noisy image"] + [f"Huber TV, α = {alpha:g}" for alpha in alpha_values],
          ["True image"] + [f"Quadratic, α = {alpha:g}" for alpha in alpha_values])
for row in range(2):
    for col in range(4):
        im = axes[row, col].imshow(images[row][col], cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_title(titles[row][col])
        axes[row, col].axis("off")
fig.colorbar(im, ax=axes, shrink=0.75, label="Image intensity")
plt.show()


### Task 4 notes

The adjoint check gives relative error $1.56\times10^{-16}$, including nonzero test values at the boundaries. With $\varepsilon=0.01$, the Huber TV runs at $\alpha=0.05,0.1,0.2$ all reached gradient RMS below $10^{-5}$ using the prescribed step size. The quadratic solves have relative normal-equation residuals below $10^{-10}$.

As $\alpha$ increases, TV removes more noise from nearly constant regions while keeping the main phantom boundaries fairly distinct; at $\alpha=0.2$ some detail also softens. At the same three $\alpha$ values, the quadratic results remain noticeably grainier. Quadratic regularization penalizes large gradients more heavily, so increasing its strength enough to remove more noise would blur intensity jumps more than TV does. This is why TV is better at preserving edges while smoothing flat regions.
